# Somaliland Hazard Monitor — test notebook

Run top to bottom. Each data source has its own cell, so if one fails you can see exactly which and why.

1. Setup  ·  2. Cyclones (JTWC)  ·  3. GDACS  ·  4. Earthquakes  ·  5. Forecasts  ·  6. River discharge  ·  7. Wildfires  ·  8. Alerts  ·  9. Map  ·  10. Save outputs

Set `USE_SAMPLE = True` to test offline with made-up data.

## 1. Setup

In [ ]:
# First time only (uncomment):
# %pip install requests folium pandas matplotlib

In [ ]:
import os, sys, json, logging
from pathlib import Path
import pandas as pd

# Make the package importable whether you open this from the project root or notebooks/
ROOT = Path.cwd() if (Path.cwd() / "somaliland_hazards").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import importlib, somaliland_hazards
from somaliland_hazards import config, alerts as alerts_mod, mapper, geo, sample
from somaliland_hazards.sources import jtwc, gdacs, earthquakes, openmeteo, firms
for mod in (config, geo, jtwc, gdacs, earthquakes, openmeteo, firms, alerts_mod, mapper, sample):
    importlib.reload(mod)   # picks up edits to the .py files without restarting

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
pd.set_option("display.max_colwidth", 80)

USE_SAMPLE = False                                   # True = offline demo data
FIRMS_KEY = os.environ.get("FIRMS_MAP_KEY", "")      # or paste your key here
status = {}

def props_df(features):
    """GeoJSON features -> DataFrame with lon/lat columns."""
    rows = [{**f["properties"], "lon": f["geometry"]["coordinates"][0],
             "lat": f["geometry"]["coordinates"][1]} for f in features
            if f["geometry"]["type"] == "Point"]
    return pd.DataFrame(rows)

if USE_SAMPLE:
    SAMPLE_DATA, _ = sample.sample_data()
print("Project root:", ROOT, "| sample mode:", USE_SAMPLE)

## 2. Tropical cyclones — JTWC
Usually empty: North Indian Ocean cyclones near Somaliland are most common in May–June and Oct–Nov.

In [ ]:
try:
    if USE_SAMPLE:
        storms = SAMPLE_DATA["storms"]
    else:
        from somaliland_hazards.http import get
        rss = get(jtwc.RSS_URL).text
        print("Warning links in RSS:", jtwc.warning_links(rss) or "none")
        storms = jtwc.fetch()
    status["JTWC"] = "ok"
except Exception as e:
    storms, status["JTWC"] = [], f"error: {e}"
    print("JTWC failed:", e)

print(f"{len(storms)} storm(s) in watch area")
for s in storms:
    print(f"{s['storm_id']} {s['name'] or ''} | {s['category']} | closest to Somaliland: {s['min_km_to_somaliland']} km")
    display(pd.DataFrame(s["track"]))

## 3. GDACS alerts

In [ ]:
try:
    gdacs_events = SAMPLE_DATA["gdacs"] if USE_SAMPLE else gdacs.fetch()
    status["GDACS"] = "ok"
except Exception as e:
    gdacs_events, status["GDACS"] = [], f"error: {e}"
    print("GDACS failed:", e)

df_gdacs = props_df(gdacs_events)
print(len(df_gdacs), "events near Somaliland")
if len(df_gdacs):
    display(df_gdacs[["hazard", "name", "alertlevel", "severity", "km_to_somaliland", "from", "to"]])

## 4. Earthquakes — USGS + EMSC

In [ ]:
try:
    if USE_SAMPLE:
        quakes = SAMPLE_DATA["quakes"]
    else:
        usgs = earthquakes.fetch_usgs(); print("USGS:", len(usgs))
        try:
            emsc = earthquakes.fetch_emsc(); print("EMSC:", len(emsc))
        except Exception as e:
            emsc = []; print("EMSC failed:", e)
        quakes = earthquakes.merge(usgs, emsc)
    status["Earthquakes"] = "ok"
except Exception as e:
    quakes, status["Earthquakes"] = [], f"error: {e}"
    print("Earthquakes failed:", e)

df_eq = props_df(quakes)
print(len(df_eq), "quakes after merge (last", config.EQ_LOOKBACK_DAYS, "days)")
if len(df_eq):
    display(df_eq[["time", "mag", "depth_km", "place", "nearest_city", "km_to_city", "source"]])

## 5. City forecasts — Open-Meteo

In [ ]:
try:
    if USE_SAMPLE:
        forecasts = SAMPLE_DATA["forecasts"]
    else:
        forecasts = openmeteo.fetch_forecast()
    status["Open-Meteo"] = "ok"
except Exception as e:
    forecasts, status["Open-Meteo"] = {}, f"error: {e}"
    print("Open-Meteo failed:", e)

df_fc = pd.DataFrame([{"city": c, **d} for c, days in forecasts.items() for d in days])
if len(df_fc):
    display(df_fc.pivot(index="city", columns="date", values="rain_mm").round(1)
            .style.background_gradient(cmap="Blues").set_caption("Daily rain (mm)"))
    display(df_fc.pivot(index="city", columns="date", values="tmax").round(0)
            .style.background_gradient(cmap="OrRd").set_caption("Max temperature (°C)"))

## 6. River discharge — GloFAS (optional)
Most Somaliland riverbeds (toggas) are seasonal, so values are often near zero. Spikes are what matter.

In [ ]:
if not USE_SAMPLE:
    try:
        rivers = openmeteo.fetch_river_discharge()
        df_q = pd.DataFrame([{"city": c, **d} for c, rows in rivers.items() for d in rows])
        display(df_q.groupby("city")[["discharge", "discharge_max"]].max()
                .rename(columns=lambda c: c + " (m³/s, 30-day max)"))
    except Exception as e:
        print("GloFAS failed:", e)
else:
    print("Skipped in sample mode")

## 7. Wildfire hotspots — NASA FIRMS
Needs a free MAP_KEY: https://firms.modaps.eosdis.nasa.gov/api/map_key/

In [ ]:
if USE_SAMPLE:
    fires = SAMPLE_DATA["fires"]; status["FIRMS"] = "ok"
elif FIRMS_KEY:
    try:
        fires = firms.fetch(FIRMS_KEY); status["FIRMS"] = "ok"
    except Exception as e:
        fires, status["FIRMS"] = [], f"error: {e}"
else:
    fires, status["FIRMS"] = [], "skipped (no key)"

df_fire = props_df(fires)
print(len(df_fire), "hotspots |", status["FIRMS"])
if len(df_fire):
    display(df_fire.groupby("nearest_city").agg(hotspots=("frp_mw", "size"),
                                                max_frp_mw=("frp_mw", "max")))

## 8. Alerts
Thresholds live in `somaliland_hazards/config.py`. You can also change them here, e.g. `config.RAIN_WATCH_MM = 20`, then re-run this cell.

In [ ]:
alerts = alerts_mod.build(storms, quakes, forecasts, fires, gdacs_events)
print("Feed status:", json.dumps(status, indent=1))
df_alerts = pd.DataFrame(alerts)
if len(df_alerts):
    colors = {"Warning": "background-color:#f5b7b1", "Watch": "background-color:#fad7a0", "Info": ""}
    display(df_alerts[["level", "hazard", "title", "detail", "source"]]
            .style.map(lambda v: colors.get(v, ""), subset=["level"]))
else:
    print("No active alerts.")

## 9. Map

In [ ]:
m = mapper.make_map(storms, quakes, forecasts, fires, gdacs_events, alerts, status,
                    sample=USE_SAMPLE)
m

## 10. Save outputs
Writes the same files the command line does. Open `output/map.html` in a browser, or share it.

In [ ]:
from somaliland_hazards import pipeline
data = {"storms": storms, "gdacs": gdacs_events, "quakes": quakes,
        "forecasts": forecasts, "fires": fires}
_, map_path = pipeline.write_outputs(data, status, out_dir=ROOT / "output", sample=USE_SAMPLE)
print("Saved:", *sorted(p.name for p in (ROOT / "output").iterdir()), sep="\n  ")

## 11. Open the full map in your browser
Double-clicking `map.html` opens it as a file, and some tile servers (OpenStreetMap especially) refuse those requests, leaving a blank basemap. This cell serves the `output` folder at `http://localhost` in the background and opens it. Run it once; re-running just reopens the page.

In [ ]:
import threading, webbrowser, functools, http.server, socketserver

PORT = 8000
if "_map_server" not in globals():
    handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=str(ROOT / "output"))
    class _Quiet(handler.func):
        def log_message(self, *a): pass
    socketserver.TCPServer.allow_reuse_address = True
    _map_server = socketserver.TCPServer(("127.0.0.1", PORT),
                                         functools.partial(_Quiet, directory=str(ROOT / "output")))
    threading.Thread(target=_map_server.serve_forever, daemon=True).start()

url = f"http://localhost:{PORT}/map.html"
webbrowser.open(url)
print("Map:", url, "  (stops when you restart the kernel)")